In [ ]:
%pip install imblearn

In [ ]:
%pip install ucimlrepo

In [84]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt 
from ucimlrepo import fetch_ucirepo
from imblearn.over_sampling import SMOTE
import seaborn as sns
from sklearn.discriminant_analysis import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from imblearn.pipeline import Pipeline  # ¡de imblearn, no de sklearn!






In [ ]:
# fetch dataset 
df = fetch_ucirepo(id=602) 

# data (as pandas dataframes) 
X = df.data.features
y = df.data.targets


In [ ]:
sns.countplot(y['Class'])  # Visualiza desbalanceo: Seker ~3000+, otras ~1000-2000
X.hist(figsize=(12,10)); plt.show()  # Distribuciones: escalas varían (Area ~10k-20k, ratios 0-1)
sns.heatmap(X.corr(), annot=False)  # Correlaciones altas (ej. Area-Perimeter)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y    #stratify para mantener proporción clases en train/test
)

scaler = StandardScaler()   # Normaliza características para mejorar rendimiento modelos sensibles a escala
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

array([[-0.3713618 , -0.53113068, -0.69536968, ...,  1.11808237,
         1.48524952,  0.95141553],
       [ 0.02895235,  0.49881788,  0.78351767, ..., -1.32149747,
        -1.87953412,  0.15816127],
       [ 0.72707783,  0.79500482,  0.82438155, ..., -0.7876742 ,
        -0.2375073 , -0.35302543],
       ...,
       [ 0.43664179,  0.87746349,  0.47987974, ..., -0.53535794,
        -0.01143779, -1.4807801 ],
       [ 0.01662606,  0.31205264,  0.64565274, ..., -1.18140133,
        -1.61703817, -0.74823181],
       [ 4.01901639,  3.47586447,  3.51587226, ..., -1.68315117,
        -0.80825296, -0.89219251]], shape=(10888, 16))

In [83]:
rf_a = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_a.fit(X_train_scaled, y_train)
y_pred_a = rf_a.predict(X_test_scaled)

print("=== RandomForest + class_weight='balanced' ===")
print(classification_report(y_test, y_pred_a))
print("Macro F1:", f1_score(y_test, y_pred_a, average="macro"))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred_a))

C:\Users\alex.iturgaitz\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


=== RandomForest + class_weight='balanced' ===
              precision    recall  f1-score   support

    BARBUNYA       0.94      0.89      0.91       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.94      0.94      0.94       326
    DERMASON       0.91      0.92      0.92       709
       HOROZ       0.97      0.95      0.96       386
       SEKER       0.94      0.96      0.95       406
        SIRA       0.86      0.85      0.85       527

    accuracy                           0.92      2723
   macro avg       0.93      0.93      0.93      2723
weighted avg       0.92      0.92      0.92      2723

Macro F1: 0.9326187597475176

Matriz de confusión:
[[236   0  17   0   1   2   9]
 [  0 104   0   0   0   0   0]
 [ 10   0 305   0   5   2   4]
 [  0   0   0 655   0  10  44]
 [  1   0   4   4 368   0   9]
 [  1   0   0   6   0 389  10]
 [  4   0   0  55   7  11 450]]


In [ ]:
smote = SMOTE(random_state=42)  # puedes ajustar k_neighbors o sampling_strategy

rf_b = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    # probar con y sin class_weight
    class_weight="balanced"
)

pipe_b = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("smote", smote),
    ("rf", rf_b),
])

pipe_b.fit(X_train, y_train)               # el Pipeline se encarga de escalar + SMOTE
y_pred_b = pipe_b.predict(X_test)          # en test NO aplica SMOTE, solo escala

print("=== SMOTE + RandomForest ===")
print(classification_report(y_test, y_pred_b))
print("Macro F1:", f1_score(y_test, y_pred_b, average="macro"))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred_b))


C:\Users\alex.iturgaitz\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


=== SMOTE + RandomForest ===
              precision    recall  f1-score   support

    BARBUNYA       0.92      0.90      0.91       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.94      0.94      0.94       326
    DERMASON       0.92      0.91      0.91       709
       HOROZ       0.96      0.95      0.96       386
       SEKER       0.94      0.96      0.95       406
        SIRA       0.86      0.86      0.86       527

    accuracy                           0.92      2723
   macro avg       0.93      0.93      0.93      2723
weighted avg       0.92      0.92      0.92      2723

Macro F1: 0.9325596766874037

Matriz de confusión:
[[239   0  15   0   1   3   7]
 [  0 104   0   0   0   0   0]
 [ 12   0 305   0   5   2   2]
 [  0   0   0 647   1  12  49]
 [  2   0   5   4 368   0   7]
 [  1   0   0   5   0 390  10]
 [  6   0   0  50   7  10 454]]
